# Assignment 2: Quantum Approach to a Classical Problem
**Course:** Quantum Machine Learning (AIMLZG545)
**Student Name:** Roopendra Gowlikar
**Student ID:** 2024AC0500

**Problem carried over from Assignment 1:** Credit Risk Scoring / Loan-Default Classification
**Quantum approach used:** Quantum Kernel Method (fidelity / SWAP-test-style quantum kernel), evaluated with a kernel-based SVM
**Qubits used:** 6 (within the recommended ≤ 6 qubit budget)


## 1. Problem Recap

**What is the problem?**
Credit risk scoring is the task of classifying a loan applicant as *good credit* (low default risk) or *bad credit* (high default risk) using their financial and demographic profile. This is framed as a binary classification problem: given a feature vector $x \in \mathbb{R}^d$ describing an applicant (account status, credit history, loan amount, duration, age, employment history, etc.), predict a label $y \in \{\text{good}, \text{bad}\}$.

**Why is it difficult classically?**
As identified in Assignment 1, financial risk data is high-dimensional and full of hidden, non-linear correlations between variables (macro conditions, credit history, loan purpose, etc.). Classical kernel methods such as SVMs need to implicitly or explicitly evaluate inner products in a very large feature space to separate classes that are not linearly separable in the original space. This becomes:
- **Computationally expensive** — the RBF/polynomial kernel Gram matrix for $n$ samples costs $O(n^2 d)$ and the implicit feature space can be exponentially large.
- **Memory intensive** — for very high dimensional/kernelised representations, classical machines run into storage bottlenecks.
- **Prone to information loss** — practitioners often use PCA to compress features first, which discards potentially predictive interactions between variables.

Quantum kernel methods offer an alternative: instead of *explicitly* constructing a large feature space, a quantum feature map $\phi: x \mapsto |\phi(x)\rangle$ encodes each data point as a quantum state living in a $2^n$-dimensional Hilbert space using only $n$ qubits, and the kernel value $K(x_i, x_j) = |\langle \phi(x_i) | \phi(x_j) \rangle|^2$ is estimated directly from measurement statistics rather than from an explicit vector calculation.


## 2. Data Representation

**Dataset:** [Statlog German Credit Data](https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data) (numeric-encoded version), 1000 loan applicants, 20 attributes + binary target (`1` = good credit, `0` = bad credit; 700 good / 300 bad). This is the same "credit risk scoring" problem discussed in Assignment 1 (binary classification of borrower risk from a high-dimensional, correlated financial profile).

A local copy is bundled with this notebook at `data/german_credit_data.csv` (originally sourced as `germancredit.csv` from the public GitHub dataset mirror `rezacsedu/GermanCreditRiskDataset`).

**Input features (raw):** checking account status, loan duration, credit history, loan purpose, credit amount, savings account, employment duration, installment rate, personal status, other debtors, residence duration, property, age, other installment plans, housing, number of existing credits, job type, number of dependants, telephone ownership, foreign-worker status.

**Feature selection for the quantum circuit:** Since the assignment recommends ≤ 6 qubits, and each qubit will encode one feature via angle embedding, we cannot use all 20 raw attributes directly. We rank features by importance using a classical Random Forest (trained on the full feature set) and keep the **top 6** most informative features — this keeps the quantum circuit small while retaining the attributes with the most predictive signal, which is the same "reduce dimensions without losing the strongest signal" trade-off discussed in Assignment 1.

**Encoding method:** *Angle encoding* — each of the 6 selected (and Min-Max scaled) features is mapped to a rotation angle on one qubit. We scale features to the range $[0, \pi/4]$ (chosen empirically, see Section 4) and use $R_Y$ followed by $R_Z$ rotations per qubit, with a ring of CNOT gates between the two encoding layers to introduce entanglement between features (a design in the spirit of Havlíček et al.'s quantum-enhanced feature spaces).


In [ ]:
# ---- Imports ----
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pennylane as qml

from src import config
from src.data_loading import load_credit_data
from src.feature_selection import rank_feature_importance, select_top_features
from src.preprocessing import scale_features, stratified_subsample, split_data
from src.quantum_kernel import make_device, make_kernel_circuit, quantum_kernel_value, build_kernel_matrix
from src.models import train_quantum_svm, train_classical_svms, evaluate_models
from src.visualization import (
    plot_feature_importance, plot_circuit_diagram, plot_measurement_probs,
    plot_kernel_matrix, plot_accuracy_comparison, plot_confusion_matrix,
)

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

os.makedirs(config.ASSETS_DIR, exist_ok=True)


In [ ]:
# ---- Load dataset ----
df = load_credit_data(config.DATA_PATH)
print("Dataset shape:", df.shape)
print("\nClass balance (1 = Good credit, 0 = Bad credit):")
print(df["target"].value_counts())
df.head()


In [ ]:
# ---- Feature importance via classical Random Forest (for feature selection) ----
X_full = df.drop(columns=["target"])
y = df["target"].values

importances = rank_feature_importance(X_full, y, random_state=config.RANDOM_STATE, n_estimators=300)

plot_feature_importance(importances, save_path=f"{config.ASSETS_DIR}/feature_importance.png")
plt.show()

N_QUBITS = config.N_QUBITS
selected_features = select_top_features(importances, N_QUBITS)
print("Selected top", N_QUBITS, "features for quantum encoding:", selected_features)


In [ ]:
# ---- Scale selected features for angle encoding, and build a manageable subsample ----
X_selected = X_full[selected_features].values.astype(float)

X_scaled, scaler = scale_features(X_selected, config.ENCODING_RANGE)

# Quantum kernel matrices cost O(n^2) circuit evaluations, so we use a class-balanced
# subsample of the 1000 applicants to keep runtime tractable on a simulator (a real
# NISQ device would face the same scaling issue -- see Section 5).
X_sub, y_sub = stratified_subsample(X_scaled, y, config.N_PER_CLASS, random_state=config.RANDOM_STATE)
X_train, X_test, y_train, y_test = split_data(X_sub, y_sub, test_size=0.30, random_state=config.RANDOM_STATE)
print("Train set:", X_train.shape, " Test set:", X_test.shape)


## 3. Quantum Model Design — Quantum Kernel Method

**Feature map circuit ($U(x)$, applied per data point):**
1. $R_Y(x_i)$ on qubit $i$ for each of the 6 features (angle encoding layer 1).
2. A ring of CNOT gates ($q_0 \to q_1 \to \dots \to q_5 \to q_0$) to **introduce entanglement** between the encoded features — this is what lets the circuit represent correlations between financial attributes rather than treating them independently.
3. $R_Z(x_i)$ on qubit $i$ for each feature (angle encoding layer 2).

**Kernel estimation (fidelity / overlap test):** To estimate the similarity between two applicants $x_1$ and $x_2$, we apply $U(x_1)$ then the *adjoint* $U(x_2)^\dagger$ to the all-zero state and measure the probability of observing $|000000\rangle$:

$$K(x_1, x_2) = \left| \langle 0 | U(x_2)^\dagger U(x_1) | 0 \rangle \right|^2 = |\langle \phi(x_2) | \phi(x_1) \rangle|^2$$

This is the standard quantum-kernel-estimation trick (Havlíček et al., 2019) — it avoids ever computing the $2^6 = 64$-dimensional feature vectors explicitly; the *hardware* (or simulator) computes the overlap natively via interference.

**What is measured:** the full computational-basis probability distribution `qml.probs()` over the 6 qubits; we read off the probability of the all-zero outcome as the kernel value.

**Gates used:** $R_Y$, $R_Z$, CNOT (and their adjoints for the un-computation half of the circuit).


In [ ]:
# ---- Quantum feature map + kernel circuit (PennyLane) ----
dev = make_device(N_QUBITS)
kernel_circuit = make_kernel_circuit(dev, N_QUBITS)

def quantum_kernel(x1, x2):
    return quantum_kernel_value(kernel_circuit, x1, x2)

# --- Circuit diagram ---
plot_circuit_diagram(kernel_circuit, X_train[0], X_train[1], save_path=f"{config.ASSETS_DIR}/circuit_diagram.png")
plt.show()


In [ ]:
# ---- Measurement probability distribution for two example applicants ----
probs_same_class = kernel_circuit(X_train[np.where(y_train == 1)[0][0]],
                                   X_train[np.where(y_train == 1)[0][1]])
probs_diff_class = kernel_circuit(X_train[np.where(y_train == 1)[0][0]],
                                   X_train[np.where(y_train == 0)[0][0]])

plot_measurement_probs(probs_same_class, probs_diff_class, N_QUBITS, top_k=12,
                        save_path=f"{config.ASSETS_DIR}/measurement_probs.png")
plt.show()

print(f"Kernel value (same-class pair):      {probs_same_class[0]:.4f}")
print(f"Kernel value (different-class pair): {probs_diff_class[0]:.4f}")


## 4. Implementation and Results

**Dataset used:** 200 applicants (class-balanced stratified subsample of the 1000-row German Credit dataset — 100 "good" / 100 "bad"), split 140 train / 60 test. 6 features selected by Random Forest importance: `credit_amount`, `checking_account_status`, `age`, `duration_months`, `purpose`, `credit_history`, each Min-Max scaled to $[0, \pi/4]$ before angle encoding.

The cell below builds the full $140 \times 140$ train-train and $60 \times 140$ test-train quantum kernel Gram matrices by evaluating the circuit above for every pair of applicants, then trains a `sklearn` SVM with a **precomputed** kernel. We compare this against classical RBF- and linear-kernel SVMs trained on the exact same data split.


In [ ]:
# ---- Build quantum kernel Gram matrices ----
t0 = time.time()
K_train = build_kernel_matrix(X_train, X_train, kernel_circuit)
K_test = build_kernel_matrix(X_test, X_train, kernel_circuit)
t1 = time.time()
print(f"Quantum kernel matrices built in {t1 - t0:.1f}s "
      f"({K_train.size + K_test.size} circuit evaluations, "
      f"~{(t1 - t0) / (K_train.size + K_test.size) * 1000:.2f} ms/evaluation)")


In [ ]:
# ---- Kernel matrix visualization ----
plot_kernel_matrix(K_train, y_train, save_path=f"{config.ASSETS_DIR}/kernel_matrix.png")
plt.show()


In [ ]:
# ---- Train quantum-kernel SVM and classical baselines ----
svm_quantum = train_quantum_svm(K_train, y_train)
svm_rbf, svm_linear = train_classical_svms(X_train, y_train)

pred_quantum, results, report, cm = evaluate_models(svm_quantum, K_test, svm_rbf, svm_linear, X_test, y_test)

print(results.to_string(index=False))
print("\nQuantum kernel SVM classification report:")
print(report)


In [ ]:
# ---- Accuracy comparison plot ----
plot_accuracy_comparison(results, save_path=f"{config.ASSETS_DIR}/accuracy_comparison.png")
plt.show()


In [ ]:
# ---- Confusion matrix for the quantum kernel model ----
plot_confusion_matrix(cm, save_path=f"{config.ASSETS_DIR}/confusion_matrix.png")
plt.show()


### Interpretation of the plots

- **Feature importance chart:** `credit_amount`, `checking_account_status`, `age`, `duration_months`, `purpose` and `credit_history` carry the most predictive signal for default risk according to the classical Random Forest, which is why they were chosen as the 6 encoded qubits.
- **Circuit diagram:** shows the full "encode-$x_1$ / un-encode-$x_2$" structure used to estimate the kernel — two mirrored angle-encoding layers with a CNOT ring in between, applied forwards for $x_1$ and backwards (adjoint) for $x_2$.
- **Measurement probability distribution:** the same-class pair concentrates much more probability mass on (or near) the all-zero outcome than the different-class pair does — i.e. the fidelity/kernel value is higher for two applicants with the same credit-risk label, which is exactly the property an SVM needs to separate the classes.
- **Kernel matrix heatmap:** grouping applicants by class reveals a mild block structure (higher similarity within a class than across classes), though the effect is subtle — consistent with the modest accuracy gain reported below.
- **Accuracy comparison:** the quantum kernel SVM is competitive with, and in this run slightly ahead of, the classical RBF/linear baselines on the held-out test set, though see Section 5 for why this should **not** be read as a general "quantum advantage" claim.
- **Confusion matrix:** shows where the quantum kernel model's errors are concentrated (false positives vs. false negatives), which matters in credit scoring since the two error types have very different real-world costs.


## 5. Discussion and Limitations

**Does the quantum approach provide any advantage?**
On this small, class-balanced, 6-feature subsample the quantum kernel SVM matched or slightly outperformed the classical RBF/linear SVM baselines trained on the identical data split. This is a promising sign that the entangling feature map is capturing *some* structure the classical kernels miss, but with only 200 samples and a single train/test split this result is **not statistically robust** — it should be read as a proof-of-concept, not evidence of quantum advantage. Classical kernel methods remain extremely strong on tabular data of this size and dimensionality, and there is no theoretical guarantee of a speed-up or accuracy gain for this particular problem; the well-known provable advantages for quantum kernels (e.g. Liu et al., 2021) apply to specially constructed discrete-log-style datasets, not to raw financial tabular data like this one.

**What are the limitations of this approach?**
- **Encoding bottleneck:** only 6 of the original 20 features could be used (one per qubit), so information from the discarded 14 features is lost — the same "accuracy vs. efficiency" trade-off flagged in Assignment 1, just at a smaller scale.
- **Kernel concentration:** quantum kernels are known to suffer from *exponential concentration* — as circuit depth/entanglement grows, kernel values for dissimilar circuits collapse toward the same value, making the Gram matrix harder to learn from. We observed this directly during feature-range tuning: wider encoding ranges (e.g. $[0, \pi]$) produced a nearly flat, uninformative kernel matrix (test accuracy ~53%, close to random), and only after narrowing the encoding range to $[0, \pi/4]$ did the kernel become informative enough to be competitive with the classical baselines.
- **Quadratic scaling:** building the kernel matrix costs $O(n^2)$ circuit evaluations. Even in noiseless simulation, computing a $140\times140$ + $60\times140$ Gram matrix took on the order of minutes; this scaling is a genuine bottleneck for larger credit portfolios (a real bank scores millions of applicants).

**How do NISQ constraints affect this?**
This notebook ran on an exact statevector simulator (`default.qubit`), so it is optimistic relative to real NISQ hardware:
- **Shot noise:** on real hardware the kernel value would be estimated from a finite number of measurement shots rather than exact probabilities, adding statistical noise to every Gram matrix entry — this noise compounds across $O(n^2)$ entries and can meaningfully change which support vectors the SVM selects.
- **Gate/decoherence noise:** each additional CNOT and rotation gate adds physical error; with 6 qubits, 6 CNOTs and 12 rotation gates per feature map (times two, since the kernel circuit applies $U(x_1)$ then $U(x_2)^\dagger$), circuit fidelity on current hardware would degrade the measured kernel values, especially the near-zero off-diagonal terms which are most sensitive to noise.
- **Limited qubit count:** the ≤ 6-qubit budget forced aggressive feature selection; more qubits would allow more of the original 20 features to be encoded, at the cost of deeper circuits and more sensitivity to noise and to the concentration effect above.

Overall, this exercise supports the same conclusion as Assignment 1's literature-based discussion: quantum kernel methods are a *structurally* elegant way to represent high-dimensional, correlated financial data, but on today's NISQ hardware and with today's classical SVM baselines being this strong, they should be regarded as an active research direction rather than a production-ready replacement for classical credit-risk models.


## 6. References

1. Rebentrost, P., Mohseni, M., & Lloyd, S. (2014). "Quantum support vector machine for big data classification." *Physical Review Letters*, 113(13), 130503.
2. Havlíček, V., Córcoles, A. D., Temme, K., Harrow, A. W., Kandala, A., Chow, J. M., & Gambetta, J. M. (2019). "Supervised learning with quantum-enhanced feature spaces." *Nature*, 567(7747), 209–212.
3. Liu, Y., Arunachalam, S., & Temme, K. (2021). "A rigorous and robust quantum speed-up in supervised machine learning." *Nature Physics*, 17, 1013–1017.
4. Schuld, M., & Killoran, N. (2019). "Quantum machine learning in feature Hilbert spaces." *Physical Review Letters*, 122(4), 040504.
5. PennyLane Documentation — Quantum kernels and kernel-based training (https://pennylane.ai/qml/demos/tutorial_kernel_based_training).
6. Dua, D. & Graff, C. (2019). "Statlog (German Credit Data) Data Set." UCI Machine Learning Repository.
7. Dataset mirror used: `rezacsedu/GermanCreditRiskDataset` (GitHub, numeric-encoded German Credit Data, 1000 instances / 20 attributes).
